In [ ]:
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.metrics import mean_absolute_error,r2_score,accuracy_score,confusion_matrix

df=pd.read_excel("week10_students.xlsx")

print(df.head())
print("dataset shape",df.shape)

  student_id                name gender               program  district  \
0    AUCA001         Aline Uwase      F   Information Systems    Gasabo   
1    AUCA002     Eric Iradukunda      M  Software Engineering   Musanze   
2    AUCA003   Diane Uwizeyimana      F            Networking   Muhanga   
3    AUCA004  Jean Bosco Ikirezi      M            Accounting  Kicukiro   
4    AUCA005    Patrick Ingabire      M   Information Systems      Huye   

   attendance_pct  quiz_avg  assignments_avg  score  
0              55      31.0             35.0   28.6  
1              73      61.1             76.7   64.5  
2              92      35.2             65.5   52.7  
3              69      65.2             54.2   68.3  
4              88      39.3             42.9   42.3  
dataset shape (150, 9)


In [ ]:
#1.1
print("number of rows and column: " , df.shape)
print("\ncorrelation with examination score:")
print(df[["quiz_avg","assignments_avg","attendance_pct","score"]].corr()["score"].drop("score"))

# the future with the largest positive correlation appears most
correlation= df[["quiz_avg","assignments_avg","attendance_pct"]].corrwith(df["score"])

print("\n most correlated feature : " , correlation.abs().idxmax())

number of rows and column:  (150, 9)

correlation with examination score:
quiz_avg           0.800726
assignments_avg    0.535983
attendance_pct     0.238118
Name: score, dtype: float64

 most correlated feature :  quiz_avg


In [ ]:
x=df[["quiz_avg","assignments_avg","attendance_pct"]]
y=df[["score"]]

x_train,x_test,y_train,y_test = train_test_split ( x,y , test_size=0.25, random_state=42)
print("training students : ", len(x_train))
print("testing students : ",len(x_test))



 #We keep 25% of the students for testing so that we can measure how well the model predicts scores for data it did not use during training. This is a fairer test than checking it on the same data it memorized.

training students :  112
testing students :  38


In [ ]:
                             #PART 2 REGRESSION
# 2.1
model = LinearRegression()
model.fit(x_train, y_train)  # train (teach) the model

print("Feature coefficients:")
for feature, coefficient in zip(x.columns, model.coef_):
    print(feature, ":", coefficient)

print("Intercept:", model.intercept_)

Feature coefficients:
quiz_avg : [0.57897954 0.364775   0.1133959 ]
Intercept: [-4.42821334]


In [ ]:
#2.2
pred=model.predict(x_test)

mae= mean_absolute_error(y_test, pred)
r2=r2_score(y_test, pred)

print("MAE",mae)
print("R2",r2)

MAE 3.329199307731651
R2 0.8866487356612142


In [ ]:
new_student = pd.DataFrame([[70,75,90]], columns=x.columns)

new_prediction = model.predict(new_student)
print("predicted score for the new student:",new_prediction[0])

my_student = pd.DataFrame([[65,70,85]],columns=x.columns)

my_prediction = model.predict(my_student)
print("predicted score using my invented values: " , my_prediction[0])

NameError: name 'pd' is not defined

In [ ]:
#3.1

df["passed"] = (df["score"] >=50).astype(int)
print("passed/fail counts (0 = fail , 1= pass) : ")
print(df["passed"].value_counts().sort_index())


xc=df[["quiz_avg","assignments_avg","attendance_pct"]]
yc=df[["passed"]]

xc_train,xc_test,yc_train,yc_test = train_test_split (xc,yc,test_size=0.25, random_state=42)


clf= LogisticRegression(max_iter=1000)
clf.fit(xc_train,yc_train)

class_pred = clf.predict(xc_test)
print("classification model.")


passed/fail counts (0 = fail , 1= pass) : 
passed
0     19
1    131
Name: count, dtype: int64
classification model.


c:\Users\SBS\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [8]:
#3.2

model_accuracy = accuracy_score(yc_test,class_pred)
baseline_accuracy = yc_test.mean()

cm=confusion_matrix(yc_test,class_pred, labels=[0,1])
print("model accuracy:" , model_accuracy)
print("everyone pass baseline accuracy:",baseline_accuracy)
print(cm)

print("\n matrix interpretation:")
print("top-left: failing student correctly identified")
print("top-right: failing student incorrectly predicted to pass ")
print("bottom-left passing students incorrectly predicted to fail:")
print("bottom-right: passing student correctly identified")


model accuracy: 0.8947368421052632
everyone pass baseline accuracy: passed    0.894737
dtype: float64
[[ 3  1]
 [ 3 31]]

 matrix interpretation:
top-left: failing student correctly identified
top-right: failing student incorrectly predicted to pass 
bottom-left passing students incorrectly predicted to fail:
bottom-right: passing student correctly identified


In [12]:
#4 feature engineering

x_quiz = df [["quiz_avg"]]
xq_train,xq_test,yq_train,yq_test = train_test_split(
  x_quiz,y, test_size =0.25 , random_state=42
)


model_quiz = LinearRegression()
model_quiz.fit(xq_train,yq_train)
r2_quiz=r2_score(yq_test, model_quiz.predict(xq_test))


x_three = df[["quiz_avg","assignments_avg","attendance_pct"]]
xt_train,xt_test,yt_train,yt_test = train_test_split(
  x_three,y , test_size = 0.25 , random_state = 42
)

model_three = LinearRegression()
model_three.fit(xt_train,yt_train)
r2_three=r2_score(yt_test,model_three.predict(x_test))


xp_program = pd.get_dummies(
  df[["quiz_avg","assignments_avg","attendance_pct","program"]],
  columns = ["program"],
  dtype = int
)

xp_train,xp_test,yp_train,yp_test = train_test_split(
  xp_program,y ,test_size = 0.25, random_state = 42
)

model_program=LinearRegression()
model_program.fit(xp_train,yp_train)
r2_program = r2_score(yp_test, model_program.predict(xp_test))

print("R2 comparison:")

print("1.quiz only: ",r2_quiz)
print("2.three numeric features: ",r2_three)
print("3. three features + program: ",r2_program)


R2 comparison:
1.quiz only:  0.6241615477396365
2.three numeric features:  0.8866487356612142
3. three features + program:  0.8804349049330737


In [13]:
#5.1 

fig1 = px.histogram(
  df,
  x="score",
  nbins=12,
  title = "1.where the class stands"

)
fig1.show()

In [ ]:


fig2 = px.scatter(
  df,
  x="quiz_avg",
  y="score",
  
  color="passed",
  hover_name="name",
  title = "2. who is at risk"
)

fig2.show()

In [ ]:


district_avg = df.groupby("district", as_index=False) ["score"].mean()

fig3 = px.bar(
  district_avg,
  x="district",
  y="score",
  title = " 3. where to act"
)

fig3.show()

In [18]:
fig2.write_html("class_dashboard.html")
print("dashboard saved as class_dashboard.html")

dashboard saved as class_dashboard.html


In [26]:
fig4=px.box(
  df,
  x="program",
  y="score",
  color= "program",
  title = "score by program"
)
fig4.show()

print("\n1. The most consistent program is information system \nbecause its scores show the smallest spread. \n2. The program with the widest spread is\n software Engeneering because its scores vary the most.")



1. The most consistent program is information system 
because its scores show the smallest spread. 
2. The program with the widest spread is
 software Engeneering because its scores vary the most.


## Model Card — Student Examination Score Prediction

**1. Purpose**

This model predicts a student's examination score using their quiz average, assignment average and attendance percentage. It is intended to help educators identify students who may need additional academic support.

**2. Data and features**

The model uses the `week10_students.csv` dataset containing 150 student records. Its input features are `quiz_avg`, `assignments_avg` and `attendance_pct`. The target is `score`, the examination score.

**3. Performance**

On the held-out test data, the model achieved an MAE of **[insert your MAE]** points and an R² of **[insert your R²]**. MAE describes the average size of prediction errors in exam points. R² describes the share of score variation explained by the model on this test set.

**4. Risks and responsible use**

A wrong prediction could cause a student to be overlooked or unfairly labelled as at risk. The dataset may contain bias or may not represent every student fairly. Predictions should be used as an early signal for supportive conversations and further review, not as a final decision about a student's ability, grades, progression or future. Teachers should consider other relevant information and speak with students before taking action.